In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import json
import pathlib
from functools import partial
from itertools import combinations

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

import gpjax as gpx

In [ ]:
cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

In [ ]:
from dmpe.utils.density_estimation import DensityEstimate

from admpe.utils.helper import get_plot_labels, pprint_helper
from admpe.data_management import (
    list_excitation_results,
    provide_excitation_results_overview,
)
from admpe.evaluation.excitation_evaluation import (
    load_excitation_comparison,
    load_corresponding_modeling_results,
)
from admpe.evaluation.excitation_visualization import (
    plot_excitation_feature_grid,
    compare_excitation_feature_grids,
    shared_density_levels,
)
from admpe.evaluation.modeling_evaluation import load_testing_sequences
from admpe.evaluation.shared import visualize_rollout_on_density
from admpe.evaluation.test_trajectories.generate_test_trajectories import TESTING_TRAJECTORIES_ROOT

---

In [ ]:
SYSTEM = "cart_pole"
LABELS = get_plot_labels(SYSTEM)

In [ ]:
testing_data = load_testing_sequences(SYSTEM)
pprint_helper(testing_data)

#### Look at the densities:

In [ ]:
# for signal_name, signal_data in testing_data.items():
#     observations = signal_data["observations"][:, :-1, :]
#     actions = signal_data["actions"]

#     observations = observations.reshape((-1, observations.shape[-1]))
#     actions = actions.reshape((-1, actions.shape[-1]))

#     testing_data_de = DensityEstimate.from_observations_actions(
#         observations=observations,
#         actions=actions,
#         points_per_dim=21, 
#         z_min=-1.05,
#         z_max=1.05,
#         bandwidth=0.1,
#     )
#     fig, axs = testing_data_de.visualize(labels=LABELS["both"])
#     fig.suptitle(signal_name)
#     fig.tight_layout()
#     plt.show()

#### Visualize testing trajectories:

In [ ]:
def load_testing_with_targets(system_name, root=TESTING_TRAJECTORIES_ROOT):
    paths = []
    seen = set()
    target_chunks = {}

    for metadata_path in sorted(pathlib.Path(root).rglob("experiment.json")):
        metadata = json.loads(metadata_path.read_text())
        if (
            metadata.get("system_name") != system_name
            or metadata.get("status") != "completed"
        ):
            continue

        path = (
            metadata_path.parent / metadata.get("dataset_file", "dataset.npz")
        ).resolve()
        if path in seen:
            continue

        seen.add(path)
        paths.append(path)

        with np.load(path, allow_pickle=False) as archive:
            n_trajectories, _, n_obs = archive["observations"].shape
            targets = (
                archive["target_observations"]
                if "target_observations" in archive.files
                else np.full((n_trajectories, n_obs), np.nan)
            )

        if targets.shape != (n_trajectories, n_obs):
            raise ValueError(f"Unexpected target shape in {path}")

        target_chunks.setdefault(metadata["signal_name"], []).append(targets)

    data = load_testing_sequences(system_name, paths=paths)
    for signal, chunks in target_chunks.items():
        data[signal]["target_observations"] = np.concatenate(chunks, axis=0)

    return data


def show_testing_densities(
    *,
    signals=None,
    trajectories=None,
    max_steps=200,
    rows_per_figure=6,
    show_targets=True,
):
    """trajectories maps signal names to one trajectory index each."""
    if max_steps is not None and max_steps < 1:
        raise ValueError("max_steps must be positive or None.")

    selected = list(testing_runs) if signals is None else list(signals)
    if not selected:
        raise ValueError("Select at least one signal.")

    runs = {name: testing_runs[name] for name in selected}
    trajectories = {} if trajectories is None else trajectories
    dim = next(iter(runs.values()))["estimator"].dim
    pairs = list(combinations(range(dim), 2))
    angle_index = {"pendulum": 0, "cart_pole": 2}.get(SYSTEM)

    figures = compare_excitation_feature_grids(
        runs,
        view="achieved",
        labels=LABELS["both"],
        levels=testing_levels,
        rows_per_figure=rows_per_figure,
        title="Testing data densities",
    )

    for page_number, (fig, axes) in enumerate(figures):
        start = page_number * rows_per_figure
        page = pairs[start : start + rows_per_figure]
        legend_ax = None

        for column, name in enumerate(selected):
            index = trajectories.get(name)
            if index is None:
                continue

            data = testing_data[name]
            observations = np.asarray(data["observations"][index])
            actions = np.asarray(data["actions"][index])
            features = np.concatenate(
                [observations[:-1], actions], axis=-1,
            )[:max_steps]

            if len(features) == 0:
                raise ValueError(f"{name}: trajectory has no actions.")

            axes[0, column].set_title(f"{name} | trajectory {index}")
            legend_ax = axes[0, column]

            for row, (i, j) in enumerate(page):
                ax = axes[row, column]

                if show_targets and "target_observations" in data:
                    target = np.asarray(data["target_observations"][index])
                
                    if np.all(np.isfinite(target)):
                        n_obs = target.size
                
                        if j < n_obs:
                            # Both axes represent observations.
                            ax.scatter(
                                target[i], target[j],
                                marker="X", s=110,
                                color="cyan", edgecolors="black",
                                linewidths=0.7, zorder=6, label="Target",
                            )
                        elif i < n_obs:
                            # Only the horizontal axis represents an observation.
                            ax.axvline(
                                target[i],
                                color="cyan", linestyle=":", linewidth=2,
                                zorder=6, label="Target",
                            )
                
                xy = features[:, [i, j]]

                # Break lines at normalized angle wrap boundaries.
                if angle_index in (i, j):
                    jumps = np.flatnonzero(
                        np.abs(np.diff(features[:, angle_index])) > 1.0
                    ) + 1
                    xy = np.insert(xy, jumps, np.nan, axis=0)

                ax.plot(
                    *xy.T, color="white", lw=1.5,
                    zorder=3, label="Trajectory",
                )

                for point, marker, size, label in (
                    (features[0], "o", 40, "Start"),
                    (features[-1], "*", 90, "End"),
                ):
                    ax.scatter(
                        point[i], point[j],
                        marker=marker, s=size,
                        color="white",
                        linewidths=0.7, zorder=4,
                    )

        if legend_ax is not None:
            handles, _ = legend_ax.get_legend_handles_labels()
            handles.extend(
            [
                Line2D(
                    [],
                    [],
                    linestyle="None",
                    marker="o",
                    markersize=6,
                    markerfacecolor="white",
                    markeredgecolor="black",
                    label="Start",
                ),
                Line2D(
                    [],
                    [],
                    linestyle="None",
                    marker="*",
                    markersize=11,
                    markerfacecolor="white",
                    markeredgecolor="black",
                    label="End",
                ),
            ])
            fig.legend(
                handles=handles,
                loc="outside lower center",
                ncol=3,
                framealpha=1,
            )

        display(fig)
        plt.close(fig)

In [ ]:
testing_data = load_testing_with_targets(SYSTEM)
pprint_helper(testing_data)

In [ ]:
testing_runs = {}

for name, data in testing_data.items():
    observations = data["observations"][:, :-1, :]
    actions = data["actions"]

    testing_runs[name] = {
        "estimator": DensityEstimate.from_observations_actions(
            observations=observations[:500].reshape(-1, observations.shape[-1]),
            actions=actions[:500].reshape(-1, actions.shape[-1]),
            points_per_dim=11,
            z_min=-1.05,
            z_max=1.05,
            bandwidth=0.1,
        )
    }

testing_levels = shared_density_levels(
    testing_runs,
    view="achieved",
    n_levels=21,
    per_pair=True,
)

In [ ]:
for index in range(20):
    show_testing_densities(
        trajectories={name: index for name in testing_data.keys()},
        max_steps=500,
        show_targets=True,
    )
    plt.show()